In [13]:
#!/usr/bin/env python3
"""
Transmit FSK with:
- FSK Modulation (Frequency-based)
- Preamble for Frame Sync

via UHD on a USRP B210.

NOTE: This is a conceptual demonstration; it's not rigorously tested or tuned.
"""

import uhd
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
import ldpc.code_util


#########################
# Configurable Settings #
#########################
TX_RATE   = 1e6   # Symbol rate (symbols per second)
FREQ      = 2.5e9  # Transmission frequency
GAIN      = 75.8    # TX gain in dB
SPS       = 20      # Samples per symbol (oversampling factor)
PREAMBLE  = [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1]  # Preamble pattern (Barker-ish)
PAYLOAD   = np.fromfile("payload.txt", dtype="c")
#######################################################


def load_base_matrix_from_text(filename):
    """
    Loads a base matrix from a text file.
    Each line corresponds to a row of integers.
    Returns a 2D NumPy array with shape (R, C).
    """
    rows = []
    
    with open(filename, 'r') as f:
        for line in f:
            # Strip whitespace, split by spaces
            str_vals = line.strip().split()
            # Convert each token to an int
            int_vals = list(map(int, str_vals))
            rows.append(int_vals)
    
    # Convert list of lists into a NumPy array
    # This assumes all rows have the same length
    base_matrix = np.array(rows, dtype=int)
    
    return base_matrix

def expand_submatrix(shift, Z):
    """
    Given a shift value and a lifting factor Z,
    return the Z x Z submatrix (in sparse format) that represents
    a cyclic shift (if shift >= 0) or a zero matrix (if shift == -1).
    """
    if shift == -1:
        # No connection => Zero submatrix
        data = []
        row = []
        col = []
        return coo_matrix((data, (row, col)), shape=(Z, Z))

    # For a shift >= 0, create a ZxZ identity matrix shifted by 'shift' columns (mod Z).
    # The (i, i+shift mod Z) = 1 for i in [0..Z-1].
    data = np.ones(Z, dtype=int)
    row = np.arange(Z)
    col = (row + shift) % Z
    return coo_matrix((data, (row, col)), shape=(Z, Z))

def create_5g_nr_parity_check_matrix(base_graph='BG1', Z=4):
    """
    Construct a 5G NR-like parity-check matrix H for demonstration.
    base_graph: 'BG1' or 'BG2'
    Z: lifting factor (must be a positive integer, typically up to 384)
    
    Returns: A scipy.sparse.coo_matrix representing the parity-check matrix.
    """
    # ------------------------------------------------
    # 1) Define a *partial* base graph from 3GPP 38.212 
    #    to keep it short. In reality, you need the full matrix.
    # 
    #    -1 means no edge
    #    0..(Z-1) means a shifted identity submatrix
    #
    #    BG1 is 46 x 68 in real 5G, BG2 is 42 x 52.
    # ------------------------------------------------
    
    if base_graph == 'BG1':
        # Partial BG1 example (4 rows x 6 cols)
        # Real BG1 is 46 x 68. This is just an example subset:
        base_graph_matrix = load_base_matrix_from_text('base_matrices/NR_1_7_60.txt')
    else:
        # Partial BG2 example (again, not the full matrix)
        # Real BG2 is 42 x 52. 
        base_graph_matrix = load_base_matrix_from_text('base_matrices/NR_1_1_384.txt')
    
    R, C = base_graph_matrix.shape
    
    # ------------------------------------------------
    # 2) Expand the base graph
    # ------------------------------------------------
    big_data = []
    big_row = []
    big_col = []
    
    for r in range(R):
        for c in range(C):
            sub_H = expand_submatrix(base_graph_matrix[r, c], Z)
            
            # The top-left corner of the submatrix in the full H
            row_offset = r * Z
            col_offset = c * Z
            
            # Convert sub_H into COO to extract data, row, col
            sub_H_coo = sub_H.tocoo()
            
            # Append to global row/col/data
            big_data.extend(sub_H_coo.data)
            big_row.extend(sub_H_coo.row + row_offset)
            big_col.extend(sub_H_coo.col + col_offset)
    
    # Final matrix size is (R*Z) x (C*Z)
    H = coo_matrix((big_data, (big_row, big_col)), shape=(R*Z, C*Z), dtype=int)
    
    return H

def gaussian_filter(num_taps, bt, sps):
    """
    Generate Gaussian filter for GFSK.
    
    Parameters:
    - num_taps: Number of filter taps (should be odd for symmetry)
    - bt: Bandwidth-Time product
    - sps: Samples per symbol
    
    Returns:
    - h: Gaussian filter coefficients (numpy array)
    """
    # Time vector centered at zero
    t = np.linspace(-num_taps//2, num_taps//2, num_taps, endpoint=False) / sps
    # Gaussian filter formula
    h = np.exp(-0.5 * (np.pi * bt * t)**2)
    # Normalize to unit energy
    h /= np.sqrt(np.sum(h**2))
    return h.astype(np.float32)

def bits_to_symbols(bit_list):
    """
    Convert list/array of bits to frequency shift symbols.
    Maps '1' to +1 and '0' to -1.
    
    Parameters:
    - bit_list: List or numpy array of bits (0 or 1)
    
    Returns:
    - symbols: Numpy array of symbols (+1 or -1)
    """
    bit_array = np.array(bit_list)

    # Validate that the bits are binary (0 or 1)
    if not np.all((bit_array == 0) | (bit_array == 1)):
        raise ValueError("All bits must be 0 or 1.")

    # Map bits to symbols: 1 -> +1.0, 0 -> -1.0
    symbols = np.where(bit_array == 1, 8.5, -8.5).astype(np.float32)

    return symbols

def string_to_bits(s):
    """
    Convert ASCII string to a list of bits (MSB first).
    
    Parameters:
    - s: Input string
    
    Returns:
    - bits: List of bits (0 or 1)
    """
    bits = []
    for char in s:
        bin_repr = format(ord(char), '08b')  # 8-bit binary
        bits.extend([int(b) for b in bin_repr])
    return bits


def fsk_modulate(bits, sps):
    """
    Perform FSK modulation on a bit stream.
    
    Parameters:
    - bits: List or numpy array of bits (0 or 1)
    - sps: Samples per symbol (oversampling factor)
    
    Returns:
    - fsk_signal: Complex baseband FSK signal
    """
    symbols = bits_to_symbols(bits)
    upsampled = np.repeat(symbols, sps) / np.sqrt(sps)
    phase =  np.cumsum(upsampled) 
    fsk_signal = np.exp(1j * phase).astype(np.complex64)
    return fsk_signal

tx_bits =None
def main():
    global tx_bits
    #------------------------------------
    # 1. Prepare Data
    #------------------------------------

    # Load the base graph matrix for 5G NR-like LDPC
    H = create_5g_nr_parity_check_matrix(base_graph='BG1', Z=11)
    G = ldpc.code_util.construct_generator_matrix(H)


    # Convert payload string to bits
    payload_bits = string_to_bits(PAYLOAD)
    # codeword = (G.T @ payload_bits[:G.shape[0]]) % 2

    
    tx_bits = PREAMBLE + list(payload_bits) + PREAMBLE[::-1]
    # FSK Modulation
    fsk_signal = fsk_modulate(tx_bits, SPS)
    # Optional: Visualize the transmitted signal

    g_filter = gaussian_filter(11, 0.95, SPS)
    # Apply Gaussian filter to the FSK signal
    gfsk_signal = np.convolve(fsk_signal, g_filter, mode='same')

    #------------------------------------
    # 2. Setup USRP
    #------------------------------------
    # Initialize USRP device (specify serial number if needed)
    usrp = uhd.usrp.MultiUSRP("serial=8000169")  # Replace with your USRP's serial or remove parameter for default
    # Transmit signal
    for i in range(2):
        usrp.send_waveform(gfsk_signal, len(gfsk_signal)/TX_RATE  , FREQ, TX_RATE, [0], GAIN)
        time.sleep(0.1)

    print("[TX] FSK signal transmitted.")

if __name__ == "__main__":
    main()


[INFO] [B200] Detected Device: B210
[INFO] [B200] Operating over USB 3.
[INFO] [B200] Initialize CODEC control...
[INFO] [B200] Initialize Radio control...
[INFO] [B200] Performing register loopback test... 
[INFO] [B200] Register loopback test passed
[INFO] [B200] Performing register loopback test... 
[INFO] [B200] Register loopback test passed
[INFO] [B200] Setting master clock rate selection to 'automatic'.
[INFO] [B200] Asking for clock rate 16.000000 MHz... 
[INFO] [B200] Actually got clock rate 16.000000 MHz.
[INFO] [B200] Asking for clock rate 32.000000 MHz... 
[INFO] [B200] Actually got clock rate 32.000000 MHz.


[TX] FSK signal transmitted.


In [14]:
from pyldpc import make_ldpc, encode

# Parameters
n = 100  # Length of the codeword
d_v = 2  # Number of ones per column in H
d_c = 4  # Number of ones per row in H

# Generate LDPC matrices
H, G = make_ldpc(n, d_v, d_c, systematic=True, sparse=True)

# Message to encode
message = np.random.randint(0, 2, G.shape[1])

# Encode the message
codeword = encode(G, message, snr=1000)

print("Message: ", message)
print("Codeword: ", codeword)


Message:  [1 1 1 0 0 0 0 1 0 1 1 0 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 0 0 1 0 1 0 1 0 0 0
 0 0 0 1 1 0 0 0 1 0 0 1 0 1]
Codeword:  [-1. -1. -1.  1.  1.  1.  1. -1.  1. -1. -1.  1. -1. -1. -1. -1. -1. -1.
 -1. -1. -1.  1. -1. -1. -1. -1. -1.  1.  1. -1.  1. -1.  1. -1.  1.  1.
  1.  1.  1.  1. -1. -1.  1.  1.  1. -1.  1.  1. -1.  1. -1.  1.  1. -1.
 -1.  1.  1.  1.  1.  1.  1. -1.  1.  1. -1. -1. -1. -1.  1. -1.  1.  1.
  1.  1. -1. -1.  1.  1.  1. -1. -1.  1.  1.  1.  1. -1.  1.  1. -1. -1.
 -1.  1. -1. -1.  1. -1.  1.  1. -1.  1.]


In [15]:
from pyldpc import decode, get_message
import numpy as np

# Simulate transmission (add noise)
snr = 5  # Signal-to-Noise Ratio in dB
noisy_codeword = codeword.copy()
noisy_codeword[0:40:4] = 1  # Add a dummy error

# Decode using Belief Propagation
decoded_codeword = decode(H, noisy_codeword, maxiter=50, snr=-10)

# Extract the message
decoded_message = get_message(G, decoded_codeword)

print("Decoded Message: ", decoded_message)


Decoded Message:  [0 1 1 0 0 0 0 1 0 1 1 0 0 1 1 1 0 1 1 1 0 0 1 1 0 1 1 0 0 1 0 1 0 1 0 0 0
 0 0 0 1 1 0 0 0 1 0 0 1 0 1]
